In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn import metrics
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
df = df.drop(["TotalCharges", "customerID"], axis=1)

In [4]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [6]:
df[df["tenure"] == 0].index

Index([488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754], dtype='int64')

In [7]:
df.drop(labels=df[df["tenure"] == 0].index, axis=0, inplace=True)

In [8]:
df[df["tenure"] == 0].index

Index([], dtype='int64')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

In [10]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [11]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1:"Yes"})

In [12]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [13]:
def object_to_int(dataframe_series):
    if dataframe_series.dtype == "object":
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series

In [14]:
df = df.apply(lambda x: object_to_int(x))
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,1


In [15]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y,  test_size=0.2, random_state=42, stratify=y)

In [17]:
num_cols = ["tenure", 'MonthlyCharges']
cat_cols_ohe = ["PaymentMethod", "Contract", "InternetService"]
cat_cols_le = list(set(X_train.columns) - set(num_cols) - set(cat_cols_ohe))

In [18]:
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [19]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Support Vector Classifier": SVC(),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(eval_metric='logloss'),
    "LightGBM": LGBMClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Gradient Boosting": GradientBoostingClassifier()
}

In [20]:
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    }

    print(f"--- {name} Results ---")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("="*40, "\n")

--- Logistic Regression Results ---
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1033
           1       0.62      0.55      0.58       374

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407


--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.61      0.49      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407


--- Decision Tree Results ---
              precision    recall  f1-score   support

           0       0.82      0.79      0.80      1033
           1       0.47      0.51      0.49       374

    accuracy                           0.72      1407
   macro avg       0.64      0

In [21]:
results_df = pd.DataFrame(results).T
print(results_df.sort_values(by="Recall", ascending=False))

                           Accuracy  Precision    Recall  F1-Score
AdaBoost                   0.788202   0.611111  0.558824  0.583799
Logistic Regression        0.791045   0.620482  0.550802  0.583569
K-Nearest Neighbors        0.764037   0.557065  0.548128  0.552561
LightGBM                   0.788913   0.619195  0.534759  0.573888
Gradient Boosting          0.790334   0.627832  0.518717  0.568082
XGBoost                    0.772566   0.581325  0.516043  0.546742
Decision Tree              0.717129   0.470443  0.510695  0.489744
Support Vector Classifier  0.786070   0.620462  0.502674  0.555391
Random Forest              0.779673   0.606667  0.486631  0.540059


In [22]:
scale_weight = 3614 / 1308

models_weighted = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "Support Vector Classifier": SVC(class_weight='balanced', random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(), 
    "XGBoost": XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_weight, random_state=42),
    "LightGBM": LGBMClassifier(class_weight='balanced', random_state=42), 
    "AdaBoost": AdaBoostClassifier(random_state=42), 
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

In [23]:
results_weighted = {}

for name, model in models_weighted.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results_weighted[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall (Churn=1)": rec,
        "F1-Score": f1
    }

    print(f"--- {name} Results ---")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("="*40, "\n")

--- Logistic Regression Results ---
              precision    recall  f1-score   support

           0       0.90      0.71      0.79      1033
           1       0.49      0.78      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407


--- Random Forest Results ---
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.49      0.55       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.78      1407


--- Decision Tree Results ---
              precision    recall  f1-score   support

           0       0.82      0.81      0.81      1033
           1       0.49      0.50      0.50       374

    accuracy                           0.73      1407
   macro avg       0.65      0

In [24]:
results_df = pd.DataFrame(results_weighted).T
print(results_df.sort_values(by="Recall (Churn=1)", ascending=False))

                           Accuracy  Precision  Recall (Churn=1)  F1-Score
Logistic Regression        0.729211   0.494078          0.780749  0.605181
Support Vector Classifier  0.729211   0.494037          0.775401  0.603538
LightGBM                   0.742715   0.511029          0.743316  0.605664
XGBoost                    0.735608   0.502024          0.663102  0.571429
AdaBoost                   0.788202   0.611111          0.558824  0.583799
K-Nearest Neighbors        0.764037   0.557065          0.548128  0.552561
Gradient Boosting          0.790334   0.627832          0.518717  0.568082
Decision Tree              0.727790   0.488312          0.502674  0.495389
Random Forest              0.784648   0.620339          0.489305  0.547085


In [25]:
lr_model = LogisticRegression(class_weight="balanced", random_state=42)

param_grid_lr = [
    {
        'penalty': ['l2'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 100], 
        'solver': ['lbfgs', 'liblinear', 'saga']
    },
    {
        'penalty': ['l1'], 
        'C': [0.001, 0.01, 0.1, 1, 10, 100], 
        'solver': ['liblinear', 'saga']
    }
]

In [26]:
grid_lr = GridSearchCV(estimator=lr_model, param_grid=param_grid_lr, cv=5, scoring="recall", n_jobs=-1, verbose=1)
grid_lr.fit(X_train, y_train)
y_pred_lr = grid_lr.predict(X_test)

print(f"Lojistik Regresyon En İyi Recall Skoru: {grid_lr.best_score_:.4f}")
print(f"Lojistik Regresyon En İyi Parametreler: {grid_lr.best_params_}\n")
print(classification_report(y_test, y_pred_lr))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Lojistik Regresyon En İyi Recall Skoru: 0.8007
Lojistik Regresyon En İyi Parametreler: {'C': 0.01, 'penalty': 'l1', 'solver': 'saga'}

              precision    recall  f1-score   support

           0       0.90      0.69      0.79      1033
           1       0.49      0.80      0.60       374

    accuracy                           0.72      1407
   macro avg       0.69      0.75      0.69      1407
weighted avg       0.79      0.72      0.74      1407



In [27]:
lgbm_model = LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1)

param_dist_lgbm = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [15, 31, 50, 63],
    'max_depth': [3, 5, 7, -1],
    'min_child_samples': [10, 20, 30, 50],
    'subsample': [0.6, 0.8, 1.0],         
    'colsample_bytree': [0.6, 0.8, 1.0]   
}

In [28]:
random_lgbm = RandomizedSearchCV(
    estimator=lgbm_model, 
    param_distributions=param_dist_lgbm, 
    n_iter=30, 
    cv=5, 
    scoring="recall", 
    random_state=42, 
    n_jobs=-1, 
    verbose=1
)
random_lgbm.fit(X_train, y_train)
y_pred_lgbm = random_lgbm.predict(X_test)

print(f"LightGBM En İyi Recall Skoru: {random_lgbm.best_score_:.4f}")
print(f"LightGBM En İyi Parametreler: {random_lgbm.best_params_}\n")
print(classification_report(y_test, y_pred_lgbm))

Fitting 5 folds for each of 30 candidates, totalling 150 fits
LightGBM En İyi Recall Skoru: 0.8094
LightGBM En İyi Parametreler: {'subsample': 0.8, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 30, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

              precision    recall  f1-score   support

           0       0.91      0.69      0.78      1033
           1       0.48      0.81      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.80      0.72      0.74      1407

